# LMCBert / context expansion — v7: journal extension experiments

Two experiments requested for the Expert Systems with Applications version, both resumable and cached on Drive:

* **Part A — WordNet (Li et al.) edge similarity**, the exact measure of Abdalgader & Al Shibli (2021), on a reduced split of the full benchmark: the 1,000 validation and 1,000 test papers plus the first `TRAIN_SUBSET` training papers (default 5,000). The same reduced split is also trained with the MiniLM and TF-IDF summaries so the three edge similarities are compared on identical data.
* **Part B — learning-rate / weight-decay sensitivity** of the main configuration (BERT on the MiniLM + Indegree summaries, full training set).

**Before running:** Runtime → Change runtime type → **T4 GPU**. Put `lmcbert_code.zip` in `MyDrive/lmcbert_colab/` (this version adds `--wd` to train.py and `--limit` to summarize.py).

Part A's WordNet summarisation is CPU-bound and slow (about 6 s per paper); the three splits run in parallel and the cell can be re-run after a disconnect, it continues where it stopped. Expect roughly 5 to 7 hours of wall time for 7,000 papers, then about 1 hour of GPU training for the three subset runs. Part B is four GPU runs of about 1 to 1.5 h each.

In [ ]:
#@title 1. Mount Drive, check GPU
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
ROOT = '/content/drive/MyDrive/lmcbert_colab'
os.makedirs(ROOT, exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout or 'NO GPU - change runtime type to T4')
assert os.path.exists(f'{ROOT}/lmcbert_code.zip'), 'Put lmcbert_code.zip in MyDrive/lmcbert_colab/ first'

In [ ]:
#@title 2. Install deps, unpack code v4, link runs/ to Drive
%pip -q install ijson gdown sentence-transformers nltk
import shutil, os
if os.path.exists('/content/lmcbert_repro'): shutil.rmtree('/content/lmcbert_repro')
os.makedirs('/content/lmcbert_repro'); os.chdir('/content/lmcbert_repro')
!unzip -q -o "$ROOT/lmcbert_code.zip" -d /content/lmcbert_repro
os.makedirs(f'{ROOT}/runs_full', exist_ok=True)
if os.path.islink('runs') or os.path.exists('runs'): os.remove('runs') if os.path.islink('runs') else shutil.rmtree('runs')
os.symlink(f'{ROOT}/runs_full', 'runs')
!grep -c "weight_decay=args.wd" train.py
!grep -c "args.limit" summarize.py
import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
#@title 3. Base data and existing summaries (all cached on Drive from earlier notebooks)
DATA = f'{ROOT}/data/splits'
GD = {'embed': f'{ROOT}/data/splits_graph_embed_indegree', 'tfidf': f'{ROOT}/data/splits_graph_tfidf_indegree'}
for d in [DATA] + list(GD.values()):
    assert all(os.path.exists(f'{d}/{s}.sum.jsonl') for s in ('train','val','test')), f'missing summaries in {d} - run notebook v3/v5 first'
!wc -l "$DATA"/*.jsonl

In [ ]:
#@title Part A, step 1. WordNet-edge summaries on the reduced split (CPU, slow, resumable; three splits in parallel)
TRAIN_SUBSET = 5000      #@param {type:"integer"}   # training papers for the subset comparison (5000 ~ 8 h CPU single-process; parallel ~5 h)
MAX_SENTS = 40           #@param {type:"integer"}   # cap on sentences per section for the WordNet graph (cost is O(n^2))
MAX_WORDS_PER_SENT = 12  #@param {type:"integer"}
WD_DIR = f'{ROOT}/data/splits_graph_wordnet_indegree'
os.makedirs(WD_DIR, exist_ok=True)
import subprocess, time
jobs = {}
for s, lim in (('val', 0), ('test', 0), ('train', TRAIN_SUBSET)):
    out = f'{WD_DIR}/{s}.sum.jsonl'
    if os.path.exists(out):
        print(s, 'already done'); continue
    cmd = ['python', '-u', 'summarize.py', '--inp', f'{DATA}/{s}.jsonl', '--out', f'{out}.part', '--backend', 'graph',
           '--sim', 'wordnet', '--centrality', 'indegree', '--resume', '--max_sents', str(MAX_SENTS),
           '--max_words_per_sent', str(MAX_WORDS_PER_SENT)] + (['--limit', str(lim)] if lim else [])
    jobs[s] = subprocess.Popen(cmd, stdout=open(f'{WD_DIR}/{s}.log', 'a'), stderr=subprocess.STDOUT)
    print('started', s)
# progress loop: prints the number of summarised papers every 5 minutes until all three finish
targets = {'val': 1000, 'test': 1000, 'train': TRAIN_SUBSET}
while any(p.poll() is None for p in jobs.values()):
    time.sleep(300)
    prog = {s: sum(1 for _ in open(f'{WD_DIR}/{s}.sum.jsonl.part')) if os.path.exists(f'{WD_DIR}/{s}.sum.jsonl.part') else 0 for s in jobs}
    print(time.strftime('%H:%M'), {s: f'{prog[s]}/{targets[s]}' for s in jobs}, flush=True)
for s, p in jobs.items():
    if p.returncode == 0:
        os.replace(f'{WD_DIR}/{s}.sum.jsonl.part', f'{WD_DIR}/{s}.sum.jsonl'); print(s, 'finished')
    else:
        print(s, 'exited with', p.returncode, '- re-run this cell to resume')
!wc -l "$WD_DIR"/*.sum.jsonl 2>/dev/null
import json
if os.path.exists(f'{WD_DIR}/val.sum.jsonl'):
    r = json.loads(open(f'{WD_DIR}/val.sum.jsonl').readline()); print(r['summary_backend']); print(r['summary'][:500])

In [ ]:
#@title Part A, step 2. Train BERT on the reduced split with WordNet, MiniLM and TF-IDF summaries (identical settings)
SEEDS_A = [42, 43, 44]   #@param
COMMON = f'--input summary --contrastive none --epochs 10 --patience 3 --batch 16 --accum 2 --max_len 512 --lr 2e-5 --fp16 --resume --skip_done --clean --progress 100 --max_train {TRAIN_SUBSET}'
SETS = {'wordnet': WD_DIR, 'embed': GD['embed'], 'tfidf': GD['tfidf']}
assert all(os.path.exists(f'{WD_DIR}/{s}.sum.jsonl') for s in ('train','val','test')), 'WordNet summaries not finished - re-run step 1'
for sim, d in SETS.items():
    for seed in SEEDS_A:
        name = f'sub{TRAIN_SUBSET}_graph_{sim}_indegree_bert_s{seed}'
        print(f'\n=================== {name} ===================', flush=True)
        !python -u train.py --data "$d" $COMMON --seed $seed --name $name
!python collect_results.py
!python stats_compare.py --a "runs/sub{TRAIN_SUBSET}_graph_tfidf_indegree_bert_s*"   --b "runs/sub{TRAIN_SUBSET}_graph_wordnet_indegree_bert_s*" --name_a "TF-IDF + Indegree (subset)"  --name_b "WordNet + Indegree (subset)"
!python stats_compare.py --a "runs/sub{TRAIN_SUBSET}_graph_wordnet_indegree_bert_s*" --b "runs/sub{TRAIN_SUBSET}_graph_embed_indegree_bert_s*"  --name_a "WordNet + Indegree (subset)" --name_b "MiniLM + Indegree (subset)"
!cp runs/stats_*.md runs/results.md runs/results.csv "$ROOT/" 

In [ ]:
#@title Part B. Learning-rate / weight-decay sensitivity of the main configuration (full training set, MiniLM + Indegree, plain BERT)
GRID = [(1e-5, 0.01), (3e-5, 0.01), (2e-5, 0.1), (1e-5, 0.1)]   # (2e-5, 0.01) is the existing run graph_embed_indegree_bert_summary_s42
BASE = f'--data "{GD["embed"]}" --input summary --contrastive none --epochs 10 --patience 3 --batch 16 --accum 2 --max_len 512 --fp16 --resume --skip_done --clean --progress 200 --seed 42'
for lr, wd in GRID:
    name = f'sweep_embed_bert_lr{lr:g}_wd{wd:g}'
    print(f'\n=================== {name} ===================', flush=True)
    !python -u train.py $BASE --lr $lr --wd $wd --name $name
!python collect_results.py
!grep -E "sweep_embed|graph_embed_indegree_bert_summary_s42" runs/results.md
!cp runs/results.md runs/results.csv "$ROOT/" 

In [ ]:
#@title Results so far
!python collect_results.py
!grep -E "sub[0-9]+_graph|sweep_embed|graph_embed_indegree_bert_summary_s42" runs/results.md
!ls "$ROOT"/stats_*.md